# 4장 2강. 구조화된 출력과 다수 문서 분석
## 실습문제


## 실습 목표

- 구조화된 출력이 필요한 이유를 설명하고 JSON의 필드를 DataFrame 컬럼으로 구분할 수 있다.
- `properties`, `type`, `enum`, `required`, `additionalProperties`를 읽고 간단한 출력 스키마를 작성할 수 있다.
- 결과의 반환 순서가 달라도 문서 ID로 원본과 결과를 연결할 수 있다.
- 출력 형식 오류, 결과 누락, API 요청 실패를 구분하여 기록할 수 있다.
- 입력·출력 건수와 ID별 대응 관계를 함께 확인하고, 구조 검증과 내용 검증을 구분할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- 아래 CSV 3개를 노트북과 같은 폴더에 둡니다. 다른 위치라면 준비 코드의 `DATA_DIR`를 수정합니다.
- 이전 강의 노트북은 필요하지 않습니다. 이 파일의 위쪽 셀부터 차례대로 실행합니다.

| 데이터 파일 | 실습에서의 역할 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건의 두 범주와 원문·기존 출력 확인 |
| 엘지유플러스_요약.csv | 동일 상담의 요약 지시·기존 출력 확인 및 분류·요약 표 구성 |
| 엘지유플러스_질의응답.csv | 질문별 입력을 구분하고 응답의 형식·누락·실패 상태 관리 |

공통 원본 컬럼은 `source_id`, `consulting_category`, `task_category`, `instruction`, `input`, `output`입니다. `source_id`는 상담 ID, `instruction`은 지시 또는 질문, `input`은 상담 원문, `output`은 CSV에 이미 저장된 출력입니다. 같은 상담에 여러 작업과 질문이 연결되므로 `source_id`가 반복됩니다.

### 실습 범위와 자료 구분

교안의 명언 사례를 사용자가 지정한 **상담 데이터**에 적용합니다. 필수 1은 출력 구조, 필수 2는 여러 문서의 결과 연결, 과제 1은 오류·누락·실패 기록을 다룹니다. 난이도는 기초~중급이며 과제도 필수 문제와 같은 수준입니다.

**이번 노트북은 실제 API를 호출하지 않습니다.** 교안에서 강조한 결과 관리·검증에 맞추어, 실제 CSV 원문과 작성자가 정한 가상 응답을 사용합니다. API 키나 유료 계정 없이 실행할 수 있습니다.

- CSV의 `output`은 기존 참고 출력이며 이번 실습에서 새로 생성한 모델 답변이 아닙니다.
- 제공하는 JSON 응답·응답 ID·요청 상태·오류는 모두 학습용 가상 자료입니다. 일부러 잘못된 응답도 포함했습니다.
- 로컬 검증 함수는 반환된 값을 검사하는 도구입니다. 스키마에 맞춰 생성을 제한하는 API의 **Structured Outputs 기능을 실행한 것은 아닙니다.**
- 원문·기존 출력·가상 응답을 별도 컬럼으로 보존합니다. 오류 행의 빈 결과를 기존 정답으로 채우지 않습니다.
- 상세 API 호출, 자동 재시도, 작업을 연결하는 워크플로우, 정확도 지표 계산은 이번 실습에 포함하지 않습니다.
- 가림 표시(`<NAME>`, `<DATE>`, `<CHARGE>` 등)는 유지합니다. 상담 내용은 해당 원문의 사실로 해석합니다.

| 교안 내용 | 대응 실습 |
|---|---|
| 구조화된 출력과 출력 스키마 | 필수 1: JSON·필수 필드·자료형·허용 범주 검증 |
| 여러 문서의 분석과 결과 연결 | 필수 2: ID 연결, 원문 보존, 중복·누락·대상 외 ID 확인 |
| 출력 오류·누락 및 처리 실패 확인 | 과제 1: 질문별 상태표, 건수 검증, 추가 확인 대상 구분 |


## 시작 전에 읽는 개념 정리

### 1. 이번에는 ‘받은 결과를 어떻게 정리하고 확인할까’를 배웁니다

앞 강의에서는 요청에 원문과 지시를 넣고 응답의 상태·텍스트·사용량을 읽었습니다. 이번에는 상담이 여러 개일 때 **결과의 모양을 일정하게 정하고, 어느 상담의 결과인지 연결하며, 제대로 확보하지 못한 결과를 구분하는 과정**을 배웁니다.

사람은 긴 답변에서 분류명과 요약을 대략 찾아 읽을 수 있습니다. 하지만 프로그램이 같은 항목을 여러 행에 정리하려면 ‘분류는 category, 요약은 summary에 넣는다’처럼 일정한 규칙이 필요합니다. 이것이 구조화된 출력을 사용하는 이유입니다. 내용을 정확하게 분석하는 일과 결과를 일정한 구조로 담는 일은 함께 필요하지만 같은 검사는 아닙니다.

| 이번 강의의 질문 | 적용하는 방법 | 결과를 읽을 때의 기준 |
|---|---|---|
| 필요한 값을 같은 위치에서 읽을 수 있는가? | 필드 이름·자료형·허용 값을 정함 | 요청한 구조를 만족하는가? |
| 어느 입력의 결과인지 알 수 있는가? | 문서 또는 작업 ID로 연결 | 각 입력이 정확히 자기 결과와 연결되는가? |
| 답을 확보하지 못한 작업은 무엇인가? | 상태와 오류 원인을 기록 | 형식 오류·결과 누락·요청 실패 중 무엇인가? |
| 정리된 결과가 사실에도 맞는가? | 원문과 작업 기준을 별도로 대조 | 구조 통과만으로 정답이라고 판단하지 않았는가? |

### 2. 주요 용어를 쉬운 말로 읽기

| 용어 | 쉬운 설명 | 이번 실습의 예 |
|---|---|---|
| 구조화된 출력 | 미리 정한 항목과 모양으로 정리한 결과 | category와 summary를 나누어 저장 |
| 필드 | 데이터에서 이름으로 구별하는 항목 | category, summary, answer |
| Key / Value | 항목 이름과 그 항목에 담긴 값 | category / 다수 요건 민원 |
| JSON | 이름·값·목록 등으로 데이터를 표현하는 형식 | 가상 응답 문자열 |
| 객체 | 이름과 값의 짝을 담은 JSON 구조 | 중괄호 안에 두 필드를 담은 결과 |
| 자료형 | 값이 문자열·숫자·불리언 등 어떤 종류인지 구분 | 요약 문장은 문자열 |
| 스키마 | 어떤 필드와 값을 허용할지 정한 규칙 | 두 필드 필수, category의 허용 범주 지정 |
| 구조 검증 | 실제 값이 정한 규칙에 맞는지 검사 | 필드 누락·자료형·범주 검사 |
| 내용 검증 | 원문과 작업 기준에 맞는 내용인지 판단 | 추후 연락을 즉시 가입으로 잘못 요약했는지 확인 |
| 문서 ID | 이번 분석의 한 입력을 구별하는 값 | 필수 2의 상담 ID, 과제의 QA01~QA05 |
| 로그 | 요청·응답·오류 등을 남긴 기록 | qa_log의 요청 상태와 원시 응답 |
| 원시 응답 | 해석하거나 정리하기 전의 응답 내용 | raw_response의 JSON 문자열 |

`category`라는 이름이 항상 같은 분류 기준을 뜻하는 것은 아닙니다. 교안의 명언 예시에서는 주제 분류였지만, 이 실습에서는 **상담 요건의 단일·다수 분류**입니다. 파일에 있는 상담 결과나 상담 주제 분류와 혼동하지 않도록, 항목 이름과 분석 기준을 함께 읽습니다.

### 3. JSON 문자열·파이썬 딕셔너리·DataFrame 구분하기

| 형태 | 어떻게 생각하면 되나요? | 이번 코드의 역할 |
|---|---|---|
| JSON 문자열 | JSON 형식으로 적혀 있는 텍스트 | 받은 응답이라고 가정한 원래 문자열 보관 |
| 파이썬 딕셔너리 | 파이썬이 이름별 값을 꺼내 쓸 수 있는 자료 | `json.loads()`로 읽은 뒤 필드 검사 |
| DataFrame | 여러 행을 같은 열 이름으로 정리한 표 | 상담별 결과와 상태를 모아 비교 |

`json.loads(문자열)`은 JSON 문자열을 파이썬 값으로 읽습니다. 이 과정을 파싱이라고 부릅니다. 반대로 `json.dumps(값)`은 파이썬 값을 JSON 문자열로 표현합니다. **읽거나 문자열로 만드는 일 자체가 API 호출은 아닙니다.** 이번 노트북에는 실제 모델에 요청하는 코드가 없습니다.

JSON 객체를 읽으면 딕셔너리, JSON 배열을 읽으면 목록, JSON의 false를 읽으면 파이썬 False가 됩니다. JSON으로 읽을 수 있다고 해서 그 결과가 항상 원하는 딕셔너리나 문자열인 것은 아닙니다. 읽은 뒤 객체 여부와 각 필드의 자료형을 다시 확인하는 이유입니다.

### 4. 스키마의 다섯 항목을 연결해서 이해하기

| 항목 | 정하는 것 | 생각할 질문 |
|---|---|---|
| `type` | 전체 값이나 특정 필드의 자료형 | 객체인가, 문자열인가? |
| `properties` | 필드별 정의 | 어떤 이름의 필드를 쓰는가? |
| `enum` | 허용하는 값의 목록 | 정해진 범주 중 하나인가? |
| `required` | 반드시 존재해야 하는 필드 | 빠진 필드는 없는가? |
| `additionalProperties` | 정의하지 않은 추가 필드 허용 여부 | 예상하지 않은 항목이 더 붙어도 되는가? |

`properties`에 필드를 적는 것과 그 필드를 반드시 포함하도록 요구하는 것은 구분합니다. 필수 여부는 `required`에서 정합니다. 또한 ‘문자열 필수’라는 규칙은 필드가 존재하고 문자열이어야 한다는 뜻이지, 문장이 유용하고 사실에 맞아야 한다는 판단까지 포함하지 않습니다.

파이썬 딕셔너리 안에서는 `False`, JSON 문자열 안에서는 `false`로 씁니다. 이것은 두 표현 방식의 표기 차이입니다. 이번 코드에서는 스키마를 파이썬 딕셔너리로 작성한 뒤 JSON 형태로 출력하므로 두 표기가 모두 보일 수 있습니다.

### 5. 생성 방식과 사후 검사를 구분하기

| 구분 | 무엇을 하는가? | 이번 실습과의 관계 |
|---|---|---|
| 프롬프트로 JSON 요청 | 원하는 출력 모양을 지시로 전달 | 형식 요청과 실제 준수 여부는 다를 수 있음을 이해 |
| API의 Structured Outputs | 지원되는 스키마에 따라 생성 구조를 제한 | 교안에서 개념을 배우며 이 노트북에서는 실행하지 않음 |
| 로컬 검사 함수 | 이미 주어진 결과가 조건에 맞는지 확인 | check_pair와 check_answer로 실제 수행 |

스키마 딕셔너리를 만들었다고 자동으로 API 설정에 적용되는 것은 아닙니다. 이번 함수들은 스키마 객체를 입력받아 해석하는 범용 검증기가 아니라, 해당 문제의 필드 조건을 직접 적은 제공 함수입니다. 따라서 스키마와 함수가 **같은 규칙을 표현하는지**를 함께 확인합니다.

### 6. ‘한 행’이 무엇인지 먼저 정하기

필수 2는 상담마다 요건 분류와 길이 제한 요약을 하나씩 선택해, 상담 한 개를 한 입력 행으로 만듭니다. 과제는 같은 상담의 여러 질문을 각각 처리하므로 질문 한 개를 한 입력 행으로 만듭니다. 같은 source_id가 반복되어도 질문이 다르면 서로 다른 작업일 수 있습니다.

`source_id`는 원본 상담, `record_id`는 원본 파일의 작업 행, `document_id`는 이번 분석 단위, `response_id`는 특정 응답을 구분합니다. 이름이 비슷해도 역할이 다르므로 어떤 단위를 구별하는 값인지 확인합니다.


## 실습 준비

아래 제공 코드를 실행하여 다음 사항을 확인하세요. 준비 함수의 구현은 평가하지 않습니다.

1. 세 파일을 읽고 컬럼·자료형, 결측치·빈 문자열을 확인합니다.
2. 고유 상담 수, 작업 유형별 행 수, 본문 글자 수의 최소·평균·최대를 확인합니다.
3. 상담 40003·40005의 원문이 세 파일에서 동일한지 확인합니다.
4. `get_record()`와 `show_record()`를 준비합니다. 질문이 여러 개라면 `instruction`까지 지정합니다.

`record_id`는 파일 이름과 **현재 파일의 0부터 시작하는 행 위치**로 만든 실습용 식별자입니다. 파일의 행 순서가 바뀌면 달라지므로 원본의 영구 ID는 아닙니다. 본문 글자 수도 API 토큰 수가 아닙니다.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])


### 준비 코드 읽기 — 파일과 표의 구조부터 확인하기

`pandas`의 `DataFrame`은 행과 열로 이루어진 표입니다. 이번 데이터의 한 행은 ‘상담 한 개’ 자체보다 **특정 상담에 대한 작업 한 개**로 이해하면 좋습니다. 같은 상담에 분류·요약·질문을 여러 개 만들 수 있기 때문입니다.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | 데이터가 다른 폴더라면 해당 폴더로 수정 |
| `DATA_DIR / "엘지유플러스_분류.csv"` | 폴더와 파일 이름을 연결 | `/`는 여기서 경로 연결 역할 |
| `pd.read_csv(...)` | CSV를 표로 읽음 | 파일 조회이며 LLM 호출이 아님 |
| `tables = {"분류": classification, ...}` | 이름과 표를 짝지어 보관 | 세 파일에 같은 점검 절차를 적용하기 위함 |
| `tables.items()` | 이름과 표를 한 쌍씩 꺼냄 | 첫 반복에서 분류 표, 다음 반복에서 다른 표 확인 |
| `table.shape` | 표의 행 수·열 수 확인 | 행 수가 곧 고유 상담 수인 것은 아님 |
| `table.dtypes.astype(str)` | 열별 자료형 이름을 문자열로 표시 | 원본 상담 본문을 변환하는 작업이 아님 |
| `table.isna().sum()` | 열별 결측치 수 계산 | 비어 있는 값 확인 |
| `table["source_id"].nunique()` | 중복을 제외한 상담 ID 개수 계산 | 동일 ID의 작업 여러 행을 한 상담으로 셈 |
| `.str.strip().eq("").sum()` | 앞뒤 공백을 제외하면 빈 문자열인 값의 수 계산 | 결측치 검사와 별도로 내용 없는 문자열 확인 |
| `.value_counts()` | 작업 유형마다 몇 행인지 계산 | 상담 수보다 작업의 구성 확인에 초점 |
| `.str.len()` | 각 행의 본문 글자 수 계산 | 토큰 수를 계산하는 기능은 아님 |
| `.agg(["min", "mean", "max"])` | 길이의 최솟값·평균·최댓값 계산 | 파일에 있는 각 행을 기준으로 집계 |
| `.round(2)` | 계산값을 소수점 둘째 자리까지 반올림 | 본문을 자르거나 요약하는 기능이 아님 |

`print()`는 값을 화면에 보여 줍니다. `f"...{값}..."`는 문자열 중간에 값을 넣는 표기입니다. 예를 들어 `{table.shape[0]:,}`의 쉼표는 큰 행 수를 천 단위로 나누어 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 출력하면서 왼쪽 인덱스를 숨기는 옵션이며, ID 열이나 실제 행을 삭제하지 않습니다.

### 같은 상담에 여러 작업이 있을 때의 ID

- `source_id`: 어느 상담에서 나온 자료인지 구분합니다.
- `task_category`: 그 상담에 대해 어떤 종류의 작업을 수행하는지 나타냅니다.
- `instruction`: 실제로 요구한 작업이나 질문 문장입니다.
- `record_id`: 현재 파일의 작업 행을 구별하기 위해 실습 코드가 추가한 이름입니다.

`[f"{name}-{i}" for i in range(len(table))]`는 현재 표의 각 행에 `분류-0`, `분류-1`처럼 이름을 붙입니다. `range(len(table))`는 0부터 행 수 직전까지의 번호를 만듭니다. 원본 표의 정렬 순서가 바뀌면 번호도 달라질 수 있으므로 원본 상담 ID와 같은 의미로 쓰지 않습니다.

준비 코드가 `record_id`를 추가한 뒤 표는 7열이 됩니다. 출력되는 ‘원본 6열’은 그 열을 추가하기 전에 확인한 값입니다. CSV 파일에 새 열을 저장하는 `to_csv()`는 실행하지 않습니다.


### 출력 검증 보조 함수 — 제공 코드

`json.dumps()`는 파이썬 딕셔너리를 JSON 문자열로 만들고, `json.loads()`는 JSON 문자열을 파이썬 값으로 읽습니다. `ensure_ascii=False`는 한글이 읽기 쉽게 출력되도록 합니다.

`check_pair()`는 필수 1·2에서 사용하는 **category·summary 두 필드만을 위한 검사 함수**입니다. JSON 문법 → 객체 여부 → 필수 필드 → 추가 필드 → 자료형 → 범주 순서로 확인하고, `(통과 여부, 이유)`를 반환합니다. 여러 문제가 있으면 처음 발견한 문제를 알려줍니다.

이 함수는 범용 JSON Schema 해석기가 아니며, 원문을 읽어 분류나 요약의 사실성을 판단하지도 않습니다. JSON Schema에 적은 조건과 코드의 검사 조건이 일치하는지를 확인하세요. 함수 구현은 제공되므로 호출하고 결과를 해석하는 데 집중합니다.

`required`와 문자열 `type`만으로는 빈 문자열을 금지하지 않습니다. 이번 함수도 빈 문자열의 의미나 요약의 충실성까지 보장하지 않습니다.


In [ ]:
import json

ALLOWED_CATEGORIES = ["단일 요건 민원", "다수 요건 민원"]

def check_pair(raw_text):
    try:
        obj = json.loads(raw_text)
    except json.JSONDecodeError:
        return False, "JSON 문법 오류"
    if not isinstance(obj, dict):
        return False, "객체가 아님"
    for key in ["category", "summary"]:
        if key not in obj:
            return False, f"필수 필드 누락: {key}"
    if set(obj) - {"category", "summary"}:
        return False, "추가 필드 존재"
    for key in ["category", "summary"]:
        if not isinstance(obj[key], str):
            return False, f"문자열이 아님: {key}"
    if obj["category"] not in ALLOWED_CATEGORIES:
        return False, "허용 범주 아님"
    return True, "구조 통과"


### 제공 함수 읽기 — check_pair의 검사 순서

이 함수의 구현을 외우는 것이 실습 목표는 아닙니다. **무엇을 검사하고, 어떤 경우에 멈추며, 무엇을 검사하지 않는지**를 이해한 뒤 제공 함수를 사용합니다.

| 순서 | 코드 | 하는 일 |
|---:|---|---|
| 1 | `json.loads(raw_text)` | JSON 문자열을 파이썬 값으로 읽기 |
| 2 | `except json.JSONDecodeError` | JSON 문법 때문에 읽지 못하면 문법 오류 반환 |
| 3 | `isinstance(obj, dict)` | 읽은 값이 딕셔너리인지 확인 |
| 4 | `key not in obj` | category와 summary가 각각 있는지 확인 |
| 5 | `set(obj) - {"category", "summary"}` | 두 이름 외에 추가된 필드 이름이 있는지 확인 |
| 6 | `isinstance(obj[key], str)` | 각 필드의 값이 문자열인지 확인 |
| 7 | `obj["category"] not in ALLOWED_CATEGORIES` | 지정한 두 범주 외의 값인지 확인 |
| 8 | `return True, "구조 통과"` | 앞 검사를 모두 통과했음을 반환 |

`isinstance(값, 자료형)`은 값의 종류를 확인합니다. `dict`는 딕셔너리, `str`은 문자열입니다. 문자열 하나를 담은 목록도 목록이라는 자료형을 가지므로 문자열과 같지 않습니다.

`set(obj)`는 딕셔너리의 **필드 이름들**을 집합으로 만듭니다. 여기에 허용한 이름들의 집합을 빼면 허용하지 않은 이름만 남습니다. 아무 이름도 남지 않으면 빈 집합이고 해당 조건문은 실행되지 않습니다. 값의 내용을 비교하는 연산은 아닙니다.

`return`을 만나면 그 함수 호출은 끝납니다. 따라서 한 응답에 여러 문제가 있어도 이 함수는 첫 번째로 발견한 문제만 알려 줍니다. ‘필수 필드 누락’이라고 출력되었다고 다른 오류 가능성이 없다고 단정할 수 없습니다. 문제가 확인된 응답은 수정 후 전체 조건을 다시 확인해야 합니다.

#### 반환값 두 개를 받는 방법

`ok, reason = check_pair(raw)`는 첫 번째 반환값을 ok에, 두 번째 반환값을 reason에 넣습니다. ok는 통과 여부인 True·False이고 reason은 사람이 읽을 수 있는 설명 문자열입니다. 이 둘을 함께 보관하면 결과가 왜 통과하거나 실패했는지 표에서 확인할 수 있습니다.

#### 이 함수가 확인하지 않는 것

함수는 상담 원문을 입력받지 않습니다. 따라서 선택한 분류가 원문에 맞는지, 요약에 핵심 사실이 빠졌는지, 사실을 반대로 적었는지는 판정할 수 없습니다. 요약이 빈 문자열이어도 문자열 자료형 조건은 통과할 수 있습니다. 이번 검사는 JSON 문자열이 주어지는 연습 자료를 전제로 하며, 모든 입력 오류를 처리하는 범용 함수로 해석하지 않습니다.


---
## 필수 1. 구조화된 출력과 스키마 확인

### 문제 1-1. 상담 분류·요약 결과의 JSON 구조 검증

#### 문제 설명

상담 **40005**의 분류와 요약 결과를 일정한 구조로 관리합니다. `category`는 **상담 요건**이며 상담 주제·상담 결과 등의 다른 분류 컬럼을 뜻하지 않습니다.

| 필드 | 자료형 | 조건 |
|---|---|---|
| category | 문자열 | 단일 요건 민원·다수 요건 민원 중 하나 |
| summary | 문자열 | 상담 내용을 요약한 값 |

두 필드는 모두 필수이며 추가 필드는 허용하지 않습니다. 이번 분류 기준에서 단일은 중심 요청·주제가 하나, 다수는 서로 구분되는 요청·주제가 둘 이상인 경우입니다. 한 요청의 절차를 여러 번 질문한 것만으로 다수로 보지는 않습니다.

아래 제공 셀에는 A~G의 가상 출력이 있습니다. A와 G를 비교할 때는 원문도 읽어야 합니다. 모든 예시는 API가 실제 반환한 값이 아닙니다.

#### 요구사항

1. `get_record(classification, 40005, "상담 요건")`과 `get_record(summarization, 40005, "길이 제한 요약")`으로 두 행을 선택하세요. 원문을 한 번 출력하고 두 파일의 지시·기존 출력을 확인하세요.
2. 위 조건을 `type`, `properties`, `enum`, `required`, `additionalProperties`를 사용한 JSON Schema 딕셔너리 `pair_schema`로 작성하세요. JSON의 `false`는 파이썬 코드에서 `False`로 씁니다.
3. 제공 함수 `check_pair()`로 A~G를 검사하여 `예시`, `구조_통과`, `확인_이유`를 가진 `structure_check` 표를 만드세요.
4. A를 `json.loads()`로 읽어 `category`와 `summary`를 각각 출력하세요.
5. A·G의 분류와 요약이 원문에 맞는지 사람이 읽고 판단하세요. 구조 검증 결과를 임의로 바꾸지 말고 내용 판단은 별도로 작성합니다.

**보조 구문:** `for name, raw in pair_cases.items():`, `ok, reason = check_pair(raw)`, `pd.DataFrame(행_목록)`

#### 해석 질문

**Q1.** 자연어로 길게 답한 결과보다 category·summary로 나눈 JSON이 DataFrame 구성에 편리한 이유는 무엇인가요?

**Q2.** 스키마의 다섯 항목은 각각 무엇을 정하나요? JSON 문법이 맞으면 스키마도 자동으로 만족하나요?

**Q3.** 프롬프트에 JSON 출력을 요청하는 것, API의 Structured Outputs, 이번 로컬 검사 함수는 각각 무엇이 다른가요? Structured Outputs를 사용하더라도 확인할 처리 상황은 무엇인가요?

**Q4.** A와 G 중 구조를 통과하는 것은 무엇이며, 내용이 더 적절한 것은 무엇인가요? 원문 표현을 근거로 답하세요.

#### 제출 결과

- 스키마·구조 검증 코드와 `structure_check`
- A의 두 필드 출력 및 A·G의 내용 판단
- Q1~Q4 답변


### 제공 자료

다음 셀은 수정하지 않고 실행하세요.


In [ ]:
# 제공 자료: 아래 JSON은 모두 작성자가 만든 가상 응답입니다.
good_summary = (
    "통화 중 대기 서비스를 삭제 후 다시 설정하고 시험 통화로 정상 작동을 확인했다. "
    "추가 결합 할인은 즉시 연결하지 않고 추후 상담 연락을 받기로 했다."
)
pair_cases = {
    "A": json.dumps({"category": "다수 요건 민원", "summary": good_summary}, ensure_ascii=False),
    "B": json.dumps({"category": "복합 문의", "summary": good_summary}, ensure_ascii=False),
    "C": json.dumps({"category": "다수 요건 민원"}, ensure_ascii=False),
    "D": json.dumps({"category": "다수 요건 민원", "summary": [good_summary]}, ensure_ascii=False),
    "E": json.dumps({"category": "다수 요건 민원", "summary": good_summary, "memo": "완료"}, ensure_ascii=False),
    "F": '{"category": "다수 요건 민원", "summary": "상담 요약",}',
    "G": json.dumps({"category": "단일 요건 민원", "summary": "고객은 추가 결합 할인 가입을 즉시 완료했다."}, ensure_ascii=False),
}
for name, raw in pair_cases.items():
    print(name, raw)


### 제공 자료를 읽는 방법

`good_summary`는 여러 줄로 나누어 적은 문자열을 하나로 연결한 값입니다. `pair_cases`는 A~G라는 사례 이름과 가상 응답 문자열을 연결한 딕셔너리입니다. `json.dumps(..., ensure_ascii=False)`는 한글이 읽히도록 JSON 문자열을 만드는 표현입니다. 한글 표시 옵션은 구조 검증 조건이 아닙니다.

일부 예시는 필드 이름·자료형·값·문법을 의도적으로 다르게 만들었습니다. 제공 셀을 정답처럼 고쳐서 검사하면 어떤 오류를 발견하는지 확인할 수 없으므로, 그대로 실행합니다. 이번 사례는 모델이 실제로 실패한 기록이 아니라 **서로 다른 오류를 구분하기 위한 가상 자료**입니다.


### 풀이 안내 — 문법·구조·내용을 나누어 적기

먼저 두 작업 행과 원문을 확인하고, 필드 조건을 스키마로 옮깁니다. 그다음 각 사례에 같은 check_pair 함수를 적용합니다. `for name, raw in pair_cases.items()`에서 name은 사례 이름이고 raw는 해당 사례의 JSON 문자열입니다.

| 확인 단계 | 답변에 적을 내용 |
|---|---|
| 문법 | JSON으로 읽을 수 있는가? |
| 구조 | 필요한 필드·자료형·범주 조건에 맞는가? |
| 내용 | 분류와 요약이 원문의 어떤 사실과 맞거나 다른가? |

내용이 틀렸다고 구조 검사 결과까지 바꾸지 않습니다. A와 G는 원문을 읽어 별도로 비교하고, ‘즉시 처리한 일’과 ‘나중에 하기로 한 일’을 구분해 근거를 작성하세요.


In [ ]:
# 문제 1 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 필수 2. 여러 상담의 결과 연결과 대응 관계 검증

### 문제 2-1. 반환 순서가 다른 분류·요약 결과를 ID로 연결하기

#### 문제 설명

상담 **40001, 40003, 40005, 40006**에 같은 분류·요약 기준을 적용한 상황입니다. 아래 제공 코드는 입력 표 `documents`와 순서가 다른 가상 결과 `returned_records`를 만듭니다.

이번에는 **문서별로 요청한 결과를 프로그램이 모아 놓은 목록**으로 가정합니다. 목록의 `document_id`는 프로그램이 원본과 연결하기 위해 보관한 정보이고, `payload` 안에는 모델이 생성했다고 가정한 category·summary만 있습니다. 따라서 필수 1의 두 필드 스키마를 그대로 사용합니다. 이 목록은 특정 API의 원본 응답 규격이 아닙니다.

각 상담에서 `상담 요건`과 `길이 제한 요약` 행을 하나씩 선택하므로, 이번 입력 표에서는 상담 ID를 `document_id`로 사용할 수 있습니다. 두 파일 전체를 source_id만으로 합치는 상황과는 다릅니다.

#### 요구사항

1. 제공 셀을 실행하여 입력 순서와 반환 순서를 확인하세요. 입력의 문서 ID가 고유한지도 확인합니다.
2. 각 payload를 `check_pair()`로 검사한 뒤 `document_id`, `category`, `summary`를 가진 `result_df`를 만드세요. 구조 오류가 있으면 무시하지 말고 확인합니다.
3. 입력 ID와 결과 ID를 비교하여 **중복 ID, 누락 ID, 대상 외 ID**를 각각 구하세요. 예: `set(입력_ID) - set(결과_ID)`는 결과가 없는 입력 ID입니다.
4. 정상 목록임을 확인한 뒤 `documents.merge(result_df, on="document_id", how="left", validate="one_to_one")`로 `mapped_results`를 만드세요. 원문·기존 분류·기존 요약과 새 category·summary를 따로 보존합니다.
5. 별도 점검 사례 `bad_result_ids`도 검사하세요. 입력과 결과가 각각 4행이라는 이유만으로 완료라고 할 수 있는지 판단하고, 중복·누락·대상 외 ID를 각각 적으세요. 이 목록을 정상 결과에 섞거나 임의로 수정하지 않습니다.

**제공 구문 설명:** `duplicated(keep=False)`는 같은 ID가 여러 번 나온 행을 모두 표시합니다. `how="left"`는 왼쪽 입력 문서를 유지합니다. `validate="one_to_one"`은 양쪽 ID가 각각 고유한지 확인하지만, 누락이나 대상 외 ID를 대신 검사해 주지는 않습니다.

#### 해석 질문

**Q1.** 이 예시에서 행 순서대로 category를 붙이면 어떤 상담의 분류가 잘못 연결되나요? ID 연결이 필요한 이유를 설명하세요.

**Q2.** 원본 CSV의 source_id, 이번 document_id, API 응답 ID는 각각 어떤 역할인가요? 질문이 여러 개인 질의응답 파일에서도 source_id만으로 한 행을 구분할 수 있나요?

**Q3.** 문서별 요청과 여러 문서를 하나의 요청에 담는 방식의 차이를 설명하세요. 각각 어떤 상황에서 고려할 수 있으며, 후자의 결과에서는 무엇을 확인해야 하나요?

**Q4.** `bad_result_ids`의 4행은 입력 4행을 모두 처리했다는 증거인가요? 중복·누락·대상 외 ID를 발견하면 어떤 정보를 확인해야 하나요?

#### 제출 결과

- 정상 목록의 구조·ID 검증 결과와 `mapped_results`
- 별도 점검 사례의 중복·누락·대상 외 ID
- Q1~Q4 답변


### 제공 자료

다음 셀은 수정하지 않고 실행하세요.


In [ ]:
# 제공 코드: 각 파일에서 작업 행을 하나씩 고른 뒤 원본을 보존합니다.
document_rows = []
for sid in [40001, 40003, 40005, 40006]:
    cr = get_record(classification, sid, "상담 요건")
    sr = get_record(summarization, sid, "길이 제한 요약")
    if cr["input"] != sr["input"]:
        raise ValueError("같은 상담의 두 원문이 다릅니다. 원본부터 확인하세요.")
    document_rows.append({
        "document_id": sid, "source_id": sid,
        "class_record_id": cr["record_id"], "summary_record_id": sr["record_id"],
        "text": cr["input"], "기존_category": cr["output"], "기존_summary": sr["output"]
    })
documents = pd.DataFrame(document_rows)

# 가상 결과입니다. 기존 output을 복사해 모델 응답으로 가장한 것이 아닙니다.
# 모든 문서에 '상담 요건 두 범주 중 하나 + 핵심 요약 문자열'이라는 같은 기준을 적용합니다.
returned_records = [
    {"document_id": 40005, "payload": {
        "category": "다수 요건 민원",
        "summary": "통화 중 대기 서비스를 재설정해 시험 통화로 확인했고, 추가 결합 할인은 추후 상담 연락을 받기로 했다."}},
    {"document_id": 40001, "payload": {
        "category": "다수 요건 민원",
        "summary": "고객은 사은품 케이스, 보험 매장 코드, 카드 할인과 휴대폰 정지 횟수를 문의하고 관련 안내를 받았다."}},
    {"document_id": 40006, "payload": {
        "category": "단일 요건 민원",
        "summary": "고객은 휴대폰 해지를 문의하고, 팩스 또는 매장 방문을 통한 신청 방법과 서류 및 처리 시점 안내를 받았다."}},
    {"document_id": 40003, "payload": {
        "category": "다수 요건 민원",
        "summary": "5G 슬림+로 요금제를 변경했으며 기존 결합 할인은 유지되고 멤버십 등급은 일반 플러스로 바뀌었다."}},
]

# 별도 점검 사례: 정상 목록과 섞지 않습니다.
bad_result_ids = pd.DataFrame({"document_id": [40005, 40001, 40001, 49999]})
print("입력 순서:", documents["document_id"].tolist())
print("반환 순서:", [r["document_id"] for r in returned_records])
print("선택한 기존 작업 결과")
print(documents[["document_id", "기존_category", "기존_summary"]].to_string(index=False))


### 제공 코드 읽기 — 원본 한 행과 결과 한 행을 연결할 준비

`for sid in [40001, 40003, 40005, 40006]`은 지정한 상담 ID를 하나씩 꺼냅니다. 각 상담에서 cr은 요건 분류 행, sr은 길이 제한 요약 행입니다. 두 원문이 다르면 `raise ValueError(...)`로 멈추고 원본을 확인하게 합니다. 서로 다른 본문을 같은 상담의 결과처럼 연결하지 않기 위한 확인입니다.

| 저장 항목 | 원래 어디에서 가져오는가? |
|---|---|
| document_id, source_id | 이번에 선택한 상담 ID |
| class_record_id | 분류 파일에서 선택한 작업 행의 식별자 |
| summary_record_id | 요약 파일에서 선택한 작업 행의 식별자 |
| text | 선택한 상담 원문 |
| 기존_category | 분류 CSV에 저장된 기존 출력 |
| 기존_summary | 요약 CSV에 저장된 기존 출력 |

각 딕셔너리를 document_rows에 모아 DataFrame으로 바꾸면 상담별 입력 표 documents가 됩니다. 이 표에서는 상담당 두 파일에서 행을 하나씩 골랐기 때문에 상담 ID가 고유한 입력 ID 역할을 할 수 있습니다. 여러 질문이 들어 있는 파일 전체를 같은 방식으로 단순히 합쳐도 된다는 뜻은 아닙니다.

returned_records는 작성자가 정한 가상 결과 목록입니다. 바깥 document_id는 프로그램이 원본과 연결하기 위해 보관한 값이고, 안쪽 payload는 분류·요약의 내용 묶음입니다. `item["payload"]["category"]`는 먼저 내용 묶음을 꺼낸 다음 그 안의 분류명을 읽는 두 단계 접근입니다. payload 안에는 category·summary만 있어 필수 1의 두 필드 검사를 그대로 적용할 수 있습니다.

이 코드의 반복문과 결과 목록에는 API 전송이 없습니다. 문서별 요청의 결과를 프로그램이 모았다고 가정한 자료입니다. 한 번의 API 요청에 여러 문서를 보내는 방식을 구현한 코드로 설명하지 않습니다.


### 풀이 안내 — 행 수와 ID 대응을 각각 확인하기

입력과 결과의 순서가 다른지 먼저 읽고, 같은 위치에 있는 행끼리 붙이지 않도록 주의합니다. 분류명이 같아 보이더라도 다른 상담의 요약이 연결될 수 있으므로 ID를 기준으로 확인해야 합니다.

| 검사 | 사용 구문 | 답변에 적을 것 |
|---|---|---|
| ID 고유성 | `.is_unique` | 한 입력을 구별하는 이름이 반복되는가? |
| 중복 결과 | `.duplicated(keep=False)` | 여러 번 나온 결과 ID는 무엇인가? |
| 누락 결과 | 입력 ID 집합에서 결과 ID 집합 빼기 | 요청했지만 결과에 없는 ID는 무엇인가? |
| 대상 외 결과 | 결과 ID 집합에서 입력 ID 집합 빼기 | 이번 입력에 없던 ID는 무엇인가? |
| 결과 연결 | `merge(..., on="document_id")` | ID가 같은 원본과 결과를 연결했는가? |

집합은 반복 횟수를 보존하지 않습니다. 집합 비교만 하지 말고 중복 검사를 함께 수행하세요. 별도 오류 목록은 진단만 하고 정상 결과에 합치거나 누락 ID를 추측해서 수정하지 않습니다.


In [ ]:
# 문제 2 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


---
## 과제 1. 질의응답의 오류·누락·요청 실패 관리

### 문제 3-1. 질문별 처리 결과를 남기고 완료 여부 판단하기

#### 문제 설명

질의응답 파일에서 아래 다섯 작업을 분석했다고 가정합니다. 같은 상담 안에 질문이 여러 개 있으므로, **질문 한 개를 처리 단위 한 행**으로 정하고 QA01~QA05라는 실습용 `document_id`를 부여합니다. 원본 source_id·record_id·질문은 그대로 보관합니다.

| document_id | source_id | 선택할 질문 |
|---|---:|---|
| QA01 | 40003 | 고객의 결합 할인은 누구 앞으로 되어 있나요? |
| QA02 | 40003 | 요금제 변경 시 초과 사용분은 어떻게 처리되나요? |
| QA03 | 40003 | 고객의 요금제 변경 시 기존 결합 할인은 유지되나요? |
| QA04 | 40005 | 고객은 해당 통신사를 이용한 지 얼마나 되었습니까? |
| QA05 | 40005 | 고객은 지금 추가 결합 할인을 받겠다고 했어? |

출력은 **answer 필드 하나를 가진 JSON 객체**입니다. answer는 필수 문자열이며 추가 필드는 허용하지 않습니다. `예/아니요형`도 이번 출력 구조에서는 문자열로 답합니다. 파이썬 `True`·`False`와 문자열 `"예"`·`"아니요"`는 다릅니다.

제공한 가상 로그에는 요청 상태와 응답 원문이 있습니다. `request_status`의 completed·failed는 실습용 기록 값입니다. None은 값이 없음을 표시하는 파이썬 값이고 실제 문자열 `"None"`이 아닙니다.

#### 요구사항

1. 제공 셀의 `qa_documents`와 `qa_log`를 확인하세요. 원본 상담은 2개지만 이번 입력 작업은 5개임을 확인합니다.
2. 위 조건을 나타내는 `answer_schema`를 작성하세요. 필수 1의 구조를 참고하되 필드는 answer 하나로 바꿉니다.
3. 입력·로그의 ID 고유성, 누락·대상 외 ID를 확인하고 ID로 연결하세요. 이 가상 로그에는 모든 요청의 기록이 있습니다. **로그 행 존재와 실제 답변 존재가 같은지**도 구분합니다.
4. 제공 함수 `check_answer()`를 이용하여 입력 5개 각각에 상태 한 개를 부여하세요. 판단 순서는 다음 표와 같습니다.

| 순서 | 조건 | status | error_reason |
|---|---|---|---|
| 1 | request_status가 failed | error | API 요청 실패 및 기록된 원인 |
| 2 | completed이지만 raw_response가 없음 | error | 분석 결과 누락 |
| 3 | JSON 또는 answer 구조 검사 실패 | error | 출력 형식 오류 및 검사 이유 |
| 4 | 위 문제가 없고 구조 통과 | success | 빈 문자열 |

5. `document_id`, `source_id`, `record_id`, `instruction`, `text`, `기존_answer`, `response_id`, `request_status`, `raw_response`, `answer`, `status`, `error_reason`을 포함한 `qa_results`를 만드세요. 실패·누락·형식 오류의 answer는 None으로 유지하고, 원본 가상 응답은 raw_response에 남깁니다.
6. 입력 5건을 success·출력 형식 오류·결과 누락·API 요청 실패로 나누어 집계하세요. `전체 입력 = 정상 처리 + 오류·누락` 관계와 질문별 한 행 보존을 확인합니다.
7. `status == "error"`인 `review_needed` 표를 만들고 각 유형의 다음 확인 사항을 적으세요. 실제 재요청은 하지 않습니다.

**보조 구문:** `for _, row in 표.iterrows():`, `row.to_dict()`, `pd.isna(값)`, `표[표["status"] == "error"]`. 제공 함수는 구현하지 않아도 됩니다.

#### 해석 질문

**Q1.** 각각의 오류 유형에 해당하는 작업 ID와 정상 처리 건수는 무엇인가요? 로그가 5행이면 분석도 완료된 것인가요?

**Q2.** QA03과 QA04는 둘 다 답변이 없는데 왜 원인을 다르게 기록하나요? 빈 값을 `아니요`, `기타`, 기존_answer로 채우면 어떤 문제가 생기나요?

**Q3.** 다시 확인하거나 처리할 때 어떤 정보를 보관해야 하나요? 5개 작업을 무조건 모두 재요청하면 안 되는 이유와 유형별 처리 방향을 설명하세요.

**Q4.** status가 success이면 답변 내용도 정답이라고 할 수 있나요? QA01·QA05를 원문과 비교하여 실제 근거를 적으세요. 정확도 지표 계산은 하지 않습니다.

#### 제출 결과

- answer 스키마·ID 검사·상태 분류 코드
- 5행의 qa_results, 유형별 집계, review_needed
- 오류 유형별 확인 사항과 Q1~Q4 답변


### 제공 자료

다음 셀은 수정하지 않고 실행하세요.


In [ ]:
# 제공 코드: 질문을 명시하여 정확한 원본 행을 고릅니다.
qa_specs = [
    ("QA01", 40003, "의문사형", "고객의 결합 할인은 누구 앞으로 되어 있나요?"),
    ("QA02", 40003, "의문사형", "요금제 변경 시 초과 사용분은 어떻게 처리되나요?"),
    ("QA03", 40003, "예/아니요형", "고객의 요금제 변경 시 기존 결합 할인은 유지되나요?"),
    ("QA04", 40005, "의문사형", "고객은 해당 통신사를 이용한 지 얼마나 되었습니까?"),
    ("QA05", 40005, "예/아니요형", "고객은 지금 추가 결합 할인을 받겠다고 했어?"),
]
qa_rows = []
for doc_id, sid, task, question in qa_specs:
    row = get_record(qa, sid, task, question)
    qa_rows.append({"document_id": doc_id, "source_id": sid, "record_id": row["record_id"],
                    "instruction": row["instruction"], "text": row["input"],
                    "기존_answer": row["output"]})
qa_documents = pd.DataFrame(qa_rows)

# 모두 가상 로그입니다. 순서가 달라도 document_id로 연결합니다.
qa_log = pd.DataFrame([
    {"document_id": "QA05", "response_id": "demo_QA05", "request_status": "completed",
     "raw_response": '{"answer": "아니요"}', "request_error": ""},
    {"document_id": "QA02", "response_id": "demo_QA02", "request_status": "completed",
     "raw_response": '{"answer": false}', "request_error": ""},
    {"document_id": "QA04", "response_id": None, "request_status": "failed",
     "raw_response": None, "request_error": "429 요청 제한(가상)"},
    {"document_id": "QA01", "response_id": "demo_QA01", "request_status": "completed",
     "raw_response": '{"answer": "남편"}', "request_error": ""},
    {"document_id": "QA03", "response_id": "demo_QA03", "request_status": "completed",
     "raw_response": None, "request_error": ""},
])

# 이번 answer 한 필드 구조만 검사하는 제공 함수입니다.
# 응답 누락과 요청 실패는 호출 전에 별도로 구분합니다.
def check_answer(raw_text):
    try:
        obj = json.loads(raw_text)
    except json.JSONDecodeError:
        return False, "JSON 문법 오류", None
    if not isinstance(obj, dict):
        return False, "객체가 아님", None
    if "answer" not in obj:
        return False, "answer 필드 누락", None
    if set(obj) - {"answer"}:
        return False, "추가 필드 존재", None
    if not isinstance(obj["answer"], str):
        return False, "answer가 문자열이 아님", None
    return True, "구조 통과", obj["answer"]

print(qa_documents[["document_id", "source_id", "instruction", "기존_answer"]].to_string(index=False))
print("\nQ4 내용 확인용 상담 40003 원문(40005는 필수 1에서 확인)")
print(qa_documents.loc[qa_documents["document_id"].eq("QA01"), "text"].iloc[0])
print("\n가상 요청 로그")
print(qa_log.to_string(index=False))


### 제공 코드 읽기 — 질문별 입력과 로그 준비

qa_specs의 각 묶음에는 **작업 ID·상담 ID·작업 유형·질문 문장**이 들어 있습니다. `for doc_id, sid, task, question in qa_specs:`는 네 값을 각 변수에 나누어 담아 한 질문씩 처리합니다. get_record에는 질문까지 전달해 정확한 원본 행을 선택합니다.

qa_documents에는 원본 상담 ID, 파일의 행 식별자, 질문, 원문, 기존 답변을 보존합니다. qa_log에는 이번 요청의 응답 ID, 요청 상태, 원시 응답, 요청 오류가 들어 있습니다. 두 표의 순서가 다르므로 document_id로 연결해야 합니다. qa_log는 실제 API 응답을 자동 수집한 자료가 아니라 제공된 가상 기록입니다.

#### check_answer는 무엇을 반환하나요?

| 확인 과정 | 통과하지 못했을 때 | 모두 통과했을 때 |
|---|---|---|
| JSON 읽기 → 객체 확인 → answer 존재 → 추가 필드 → 문자열 검사 | False, 오류 설명, None | True, 구조 통과, 읽은 answer 값 |

check_pair와 달리 반환값이 세 개입니다. 첫 번째는 통과 여부, 두 번째는 설명, 세 번째는 통과한 경우의 실제 답변입니다. `ok, detail, parsed_answer = check_answer(...)`처럼 받으면 각 역할을 구분할 수 있습니다. 함수는 답변 내용이 사실에 맞는지 판단하지 않습니다.

요청 실패와 raw_response 누락은 이 함수에 전달하기 전에 구분합니다. 이 함수는 JSON 문자열을 검사하도록 제공되어 있으므로 응답이 없는 None을 그대로 넣어 처리하려고 하지 않습니다.

#### False·None·문자열을 구분하기

| 표현 | 뜻 |
|---|---|
| JSON의 `false` | 불리언 값이며 파이썬으로 읽으면 False |
| 파이썬의 `None` | 값이 없음을 표시 |
| 문자열 `"아니요"` | 글자로 적힌 부정 답변 |
| 문자열 `"None"` | N·o·n·e 네 글자가 있는 문자열로, None 값과 다름 |

JSON의 false를 읽었다고 질문에 대한 올바른 ‘아니요’ 답변이 확보된 것은 아닙니다. 질문이 무엇인지와 요구한 자료형을 함께 확인합니다. 문자열 조건만 통과한 답변도 원문 대조가 필요합니다.


### 풀이 안내 — 원인이 사라지지 않도록 검사 순서를 지키기

입력과 로그의 ID를 먼저 확인한 뒤 같은 작업끼리 연결합니다. 이후 각 행에 대해 요청 실패인지, 완료 기록은 있지만 응답이 없는지, 응답이 구조를 어겼는지를 순서대로 확인합니다. 요청 실패에도 응답이 없을 수 있으므로, 빈 값부터 검사하면 서로 다른 원인을 같은 누락으로 기록할 수 있습니다.

| 단계 | 작성할 결과 |
|---|---|
| 행별 판단 | status 한 개, error_reason 한 개 |
| 정상 구조 통과 | 파싱한 답변을 answer에 기록 |
| 오류·누락·실패 | answer는 None으로 두고 원시 응답과 원인은 보존 |
| 전체 집계 | 정상·형식 오류·누락·요청 실패 건수를 각각 확인 |
| 추가 확인 | error 행만 선택하여 다음 확인 사항 작성 |

`for _, row in 표.iterrows()`는 표에서 인덱스와 행을 하나씩 꺼내는 표현입니다. 여기서 `_`는 이번 코드에서 쓰지 않는 인덱스를 받는 변수 이름입니다. `row.to_dict()`로 기존 행 정보를 복사하고, `update()`로 판단 결과를 추가하면 원문과 로그 정보를 함께 보존할 수 있습니다.

상태 기록이 다섯 행 있다고 정상 답변도 다섯 개라는 뜻은 아닙니다. 개수 검증과 ID별 상태 확인을 함께 하고, 원문에 맞는 답인지도 별도로 설명하세요. 자동 재요청이나 정확도 계산은 과제에 포함되지 않습니다.


In [ ]:
# 문제 3 풀이 코드를 작성하세요.


### 해석 답안 작성

- Q1.
- Q2.
- Q3.
- Q4.


### 개념 다시 정리 — 통과했다는 말의 범위를 붙이기

| 확인 결과 | 확인한 사실 | 아직 별도로 볼 것 |
|---|---|---|
| JSON 파싱 성공 | 문자열을 JSON 값으로 읽을 수 있음 | 원하는 객체·필드·자료형인지 |
| 구조 통과 | 제공 함수의 구조 조건을 만족 | 원문과 분석 기준에 맞는 내용인지 |
| 입력·결과 행 수 일치 | 표의 길이가 같음 | ID 중복·누락·대상 외 결과 |
| ID 대응 정상 | 입력과 결과의 연결 관계가 맞음 | 각 행의 답변 존재·처리 상태 |
| 로그 행 존재 | 해당 작업의 기록이 남아 있음 | 실제 응답과 정상 답변을 확보했는지 |
| status가 success | 이번 처리·구조 검사 기준을 통과 | 사실 관계와 답변의 유용성 |

해석할 때는 “정상이다”만 쓰기보다 “어떤 검사를 통과했으며 무엇은 아직 확인해야 한다”라고 적습니다. 실패한 결과를 기존 정답이나 임의의 부정 답변으로 채우지 않고 원인을 남기는 것도 정확한 결과 관리의 일부입니다.


---
## 실습 마무리

아래 질문에 각각 2~3문장으로 답하세요.

1. **어떤 분석 문제가 있었는가?** 출력 구조 차이, 순서 변경, 중복·누락·요청 실패를 정리하세요.
2. **어떤 방법을 적용했는가?** 스키마, JSON 검사, 문서 ID 연결, 상태 기록을 설명하세요.
3. **무엇을 근거로 결론을 내렸는가?** 필드·자료형·허용 값, ID 목록과 건수, 요청 기록, 원문을 구분하여 설명하세요.

### 제출 전 확인

- 필수 1·필수 2·과제 1의 코드와 Q1~Q4 답변을 모두 작성했나요?
- 원문과 기존 출력, 가상 응답을 별도로 유지했나요?
- 개수뿐 아니라 ID별 중복·누락·대상 외 결과를 확인했나요?
- error 행의 answer를 임의의 값으로 채우지 않았나요?
- 구조 통과와 내용의 정확성을 구분했나요?


### 마무리 답안

1.
2.
3.
